## WSW Gates - Beach Courses

Generate beach courses for Weymouth Speed Week

In [1]:
import os
import sys

import jinja2

import pyproj
from vincenty import vincenty

### SWCP Waypoints

South West Coast Path waypoints identified on Google Earth

In [2]:
# OTC
waypoint1 = (50.57482500, -2.46500000)

# Billy Winters
waypoint2 = (50.57890556, -2.46826944)

### Jinja Setup

Prepare environment for Jinja templates

In [3]:
projdir = os.path.realpath(os.path.join(sys.path[0], '..'))

coursesPath = os.path.join(projdir, 'courses')

gtxPath = os.path.join(coursesPath, 'gtx')
kmlPath = os.path.join(coursesPath, 'kml')

templateLoader = jinja2.FileSystemLoader([gtxPath, kmlPath])

templateEnv = jinja2.Environment(loader=templateLoader,
                                 autoescape=True, trim_blocks=True, lstrip_blocks=True)

### Calculate distance using Vincenty

The vincenty library returns distances in km, hence the multiplication by 1000

In [4]:
distance = 1000 * vincenty(waypoint1, waypoint2)

print(f"Distance: {distance} meters")

Distance: 509.587 meters


### Calculate distance and azimuths using pyproj

The pyproj library returns distances in metres, and azimuths betwen -180 and +180

In [5]:
geod = pyproj.Geod(ellps='WGS84')

In [6]:
# Use SWCP waypoints
lat1, lon1 = waypoint1
lat2, lon2 = waypoint2

# Determine forward and back azimuths, plus distance between the waypoints
forward_azimuth, back_azimuth, distance = geod.inv(lon1, lat1, lon2, lat2)

# Convert negative values to positive values
forward_azimuth = (forward_azimuth + 360) % 360
back_azimuth = (back_azimuth + 360) % 360
line_azimuth = (forward_azimuth + 90)  % 360

# Report the results
print(f"Distance: {distance:.3f} meters")
print(f"Heading: {forward_azimuth:.3f} degrees")

Distance: 509.587 meters
Heading: 332.971 degrees


### Generate KML

Use Jinja to generate KML from template

In [7]:
template = templateEnv.get_template("template.kml")

kml = template.render()

### Generate Gates

Use Jinja to generate .gtx file from template

In [8]:
interval = 50

w1 = geod.fwd_intermediate(lon1, lat1, back_azimuth, npts=6, del_s=interval, initial_idx=0,
                           return_back_azimuth=True)
w2 = geod.fwd_intermediate(lon2, lat2, back_azimuth, npts=6, del_s=interval, initial_idx=0,
                           return_back_azimuth=True)

In [9]:
track_length = -500
gate_width = 1000
i = 0

c1_lon = w1.lons[i]
c1_lat = w1.lats[i]
c2_lon, c2_lat, ignore = geod.fwd(c1_lon, c1_lat, line_azimuth, gate_width)
start_lon, start_lat, ignore = geod.fwd(c1_lon, c1_lat, line_azimuth, gate_width / 2)

c3_lon = w2.lons[i]
c3_lat = w2.lats[i]
c4_lon, c4_lat, ignore = geod.fwd(c3_lon, c3_lat, line_azimuth, gate_width)
finish_lon, finish_lat, ignore = geod.fwd(c3_lon, c3_lat, line_azimuth, gate_width / 2)

corners_lat_lon = "{:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f}".format(
    c1_lat, c1_lon, c2_lat, c2_lon, c3_lat, c3_lon, c4_lat, c4_lon)

template = templateEnv.get_template("template.gtx")
gtx = template.render(track_length=track_length, gate_width=gate_width,
                      start_lat=round(start_lat, 7), start_lon=round(start_lon, 7),
                      finish_lon=round(finish_lon, 7), finish_lat=round(finish_lat, 7),
                      corners_lat_lon=corners_lat_lon
                     )

gtxFile = os.path.join(gtxPath, 'test.gtx')
with open(gtxFile, 'w', encoding='utf-8') as f:
	f.write(gtx)

name = 'Weymouth Speed Week'

polygon_coordinates = \
    "{:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0".format(
    c1_lon, c1_lat, c3_lon, c3_lat, c4_lon, c4_lat, c2_lon, c2_lat, c1_lon, c1_lat)

template = templateEnv.get_template("template.kml")
kml = template.render(name=name,
                      start_lat=round(start_lat, 7), start_lon=round(start_lon, 7),
                      finish_lon=round(finish_lon, 7), finish_lat=round(finish_lat, 7),
                      polygon_coordinates=polygon_coordinates
                     )

kmlFile = os.path.join(kmlPath, 'test.kml')
with open(kmlFile, 'w', encoding='utf-8') as f:
	f.write(kml)